# 00 — Exploración inicial

Carga rápida de los 4 CSVs ya en repo y primer `value_counts()` por dataset.
Notebook para Persona C — punto de partida del feature engineering.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from packages.data.paths import RAW_AIRE, RAW_METEO, RAW_HOSPITALES, RAW_TRAFICO, BCN_BBOX

pd.set_option('display.max_columns', 50)
REPO_ROOT

## 1. Aire (Open-Meteo AQ single-point)

In [ ]:
aire = pd.read_csv(RAW_AIRE, parse_dates=['time'])
print(aire.shape)
aire.head()

## 2. Meteo (Open-Meteo Forecast single-point)

In [ ]:
meteo = pd.read_csv(RAW_METEO, parse_dates=['time'])
print(meteo.shape)
meteo.head()

## 3. Hospitales (Overpass dump — SUCIO, filtrar)

Incluye nodos de Granollers, Sant Boi, e incluso Venezuela (lat negativa). Aplicamos bbox BCN.

In [ ]:
hosp = pd.read_csv(RAW_HOSPITALES)
print('Total:', len(hosp))
min_lat, max_lat, min_lon, max_lon = BCN_BBOX
in_bcn = hosp['lat'].between(min_lat, max_lat) & hosp['lon'].between(min_lon, max_lon)
print('En BCN:', in_bcn.sum())
hosp[in_bcn].head(10)

## 4. Tráfico TRAMS (mayo 2026)

Estado por tramo cada ~5 min. `estatActual ∈ {1..6}`: 1=fluido…5=congestión, 6=cortado.

In [ ]:
trafico = pd.read_csv(RAW_TRAFICO)
trafico['timestamp'] = pd.to_datetime(trafico['data'].astype(str), format='%Y%m%d%H%M%S')
print(trafico.shape)
print('Tramos únicos:', trafico['idTram'].nunique())
print('Rango fechas:', trafico['timestamp'].min(), '→', trafico['timestamp'].max())
trafico.head()

In [ ]:
# Distribución de estados — sanity check
trafico['estatActual'].value_counts().sort_index()

## 5. Patrón hora·día_semana del tráfico

Base para la heurística si el LightGBM no llega a tiempo.

In [ ]:
import numpy as np
trafico['hour'] = trafico['timestamp'].dt.hour
trafico['dow'] = trafico['timestamp'].dt.dayofweek  # 0=lunes
perfil = trafico.pivot_table(
    index='hour', columns='dow', values='estatActual', aggfunc='mean'
)
perfil.round(2)

## TODO siguientes

1. Descargar GeoJSON 73 barrios y tramos viarios de Open Data BCN.
2. Mapear tramos → barrio (centroide dentro de polígono).
3. Cargar dataset Kaggle accidentes y validar mapping distrito↔barrio.
4. Extender meteo/aire a 73 puntos (centroides de barrio).